# Generate Time Series Cache Data for StockMapper

This notebook fetches historical stock data from Yahoo Finance and generates cached JSON files that the StockMapper app can use in CANNED_DATA mode.

## What This Does

For each stock in `stocks.csv`, this notebook will generate three types of cached time series data:

1. **Intraday (1 day)**: Recent 1-day price data at 5-minute or hourly intervals
2. **5-day**: Recent 5 days of intraday data  
3. **Daily**: Long-term historical daily data (1-2 years)

Each cache file will be saved in the format:
```
public/data/nyse/cache/{symbol}_{type}.json
```

Where `{type}` is one of: `intraday`, `5day`, or `daily`.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import json
from pathlib import Path
from datetime import datetime, timedelta
import time
from tqdm import tqdm  # For progress bars

print("✓ Libraries imported successfully")

: 

## 2. Setup Paths and Configuration

In [ ]:
# Define paths
BASE_DIR = Path('/Users/hermannair/Documents/GitHub/stockmapper')
DATA_DIR = BASE_DIR / 'public' / 'data' / 'nyse'
STOCKS_CSV = DATA_DIR / 'stocks.csv'
CACHE_DIR = DATA_DIR / 'cache'

# Create cache directory if it doesn't exist
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
RATE_LIMIT_DELAY = 0.5  # Seconds between API calls to avoid rate limiting
MAX_RETRIES = 3  # Number of retries for failed API calls

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Cache directory: {CACHE_DIR}")
print(f"✓ Cache directory exists: {CACHE_DIR.exists()}")

## 3. Load Stock Symbols from CSV

In [ ]:
# Load the stocks CSV file
stocks_df = pd.read_csv(STOCKS_CSV)

print(f"✓ Loaded {len(stocks_df)} stocks from {STOCKS_CSV.name}")
print(f"\nFirst 5 stocks:")
print(stocks_df[['Name', 'Symbol']].head())
print(f"\nTotal unique symbols: {stocks_df['Symbol'].nunique()}")

# Get list of all symbols
all_symbols = stocks_df['Symbol'].tolist()

# Optional: For testing, start with a small subset
# Uncomment the line below to test with just 10 stocks first
# all_symbols = all_symbols[:10]

print(f"\n✓ Will generate cache for {len(all_symbols)} symbols")

## 4. Define Data Formatting Functions

The StockMapper app expects time series data in a specific JSON format:

```json
{
  "headers": ["t", "price", "volume"],
  "data": [
    [timestamp_ms, price, volume],
    [timestamp_ms, price, volume],
    ...
  ]
}
```

Where:
- `t` = timestamp in milliseconds since epoch
- `price` = close price
- `volume` = trading volume

In [ ]:
def format_timeseries_data(hist_df):
    """
    Convert yfinance history DataFrame to StockMapper format.
    
    Args:
        hist_df: pandas DataFrame from yfinance with columns [Open, High, Low, Close, Volume]
    
    Returns:
        dict with 'headers' and 'data' arrays in StockMapper format
    """
    if hist_df is None or hist_df.empty:
        return None
    
    data_rows = []
    for timestamp, row in hist_df.iterrows():
        # Convert timestamp to milliseconds since epoch
        ts_ms = int(timestamp.timestamp() * 1000)
        price = round(row['Close'], 2)
        volume = int(row['Volume']) if not pd.isna(row['Volume']) else 0
        
        data_rows.append([ts_ms, price, volume])
    
    return {
        "headers": ["t", "price", "volume"],
        "data": data_rows
    }

# Test with a sample
print("✓ Data formatting function defined")

## 5. Fetch Time Series Data Functions

In [ ]:
def fetch_intraday_data(symbol, period='1d', interval='5m'):
    """
    Fetch intraday data (1 day at 5-minute intervals).
    
    Args:
        symbol: Stock symbol (e.g., 'AAPL')
        period: Time period ('1d' for 1 day)
        interval: Data interval ('5m', '15m', '1h', etc.)
    
    Returns:
        Formatted time series data dict or None if failed
    """
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period=period, interval=interval)
        
        if hist.empty:
            return None
        
        return format_timeseries_data(hist)
    except Exception as e:
        print(f"    ✗ Error fetching intraday for {symbol}: {e}")
        return None


def fetch_5day_data(symbol):
    """
    Fetch 5-day intraday data.
    
    Args:
        symbol: Stock symbol (e.g., 'AAPL')
    
    Returns:
        Formatted time series data dict or None if failed
    """
    try:
        ticker = yf.Ticker(symbol)
        # Use hourly interval for 5 days to reduce data size
        hist = ticker.history(period='5d', interval='1h')
        
        if hist.empty:
            return None
        
        return format_timeseries_data(hist)
    except Exception as e:
        print(f"    ✗ Error fetching 5day for {symbol}: {e}")
        return None


def fetch_daily_data(symbol, period='1y'):
    """
    Fetch daily historical data.
    
    Args:
        symbol: Stock symbol (e.g., 'AAPL')
        period: Time period ('1y', '2y', '5y', etc.)
    
    Returns:
        Formatted time series data dict or None if failed
    """
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period=period, interval='1d')
        
        if hist.empty:
            return None
        
        return format_timeseries_data(hist)
    except Exception as e:
        print(f"    ✗ Error fetching daily for {symbol}: {e}")
        return None


print("✓ Data fetching functions defined")

## 6. Save Cache Files Function

In [ ]:
def save_cache_file(symbol, data_type, data):
    """
    Save time series data as JSON cache file.
    
    Args:
        symbol: Stock symbol (e.g., 'AAPL')
        data_type: Type of data ('intraday', '5day', 'daily')
        data: Formatted time series data dict
    
    Returns:
        Path to saved file or None if failed
    """
    if data is None:
        return None
    
    try:
        filename = f"{symbol}_{data_type}.json"
        filepath = CACHE_DIR / filename
        
        with open(filepath, 'w') as f:
            json.dump(data, f, separators=(',', ':'))  # Compact JSON
        
        return filepath
    except Exception as e:
        print(f"    ✗ Error saving cache for {symbol} {data_type}: {e}")
        return None


print("✓ Cache saving function defined")

## 7. Process Single Stock Function

In [ ]:
def process_stock(symbol, retry_count=0):
    """
    Fetch and cache all time series data for a single stock.
    
    Args:
        symbol: Stock symbol to process
        retry_count: Current retry attempt number
    
    Returns:
        dict with results for each data type
    """
    results = {
        'symbol': symbol,
        'intraday': None,
        '5day': None,
        'daily': None,
        'success': False
    }
    
    try:
        print(f"  Processing {symbol}...")
        
        # Fetch intraday data
        intraday_data = fetch_intraday_data(symbol)
        if intraday_data:
            save_cache_file(symbol, 'intraday', intraday_data)
            results['intraday'] = True
            print(f"    ✓ Intraday: {len(intraday_data['data'])} data points")
        else:
            print(f"    - Intraday: No data")
        
        time.sleep(RATE_LIMIT_DELAY)
        
        # Fetch 5-day data
        day5_data = fetch_5day_data(symbol)
        if day5_data:
            save_cache_file(symbol, '5day', day5_data)
            results['5day'] = True
            print(f"    ✓ 5-day: {len(day5_data['data'])} data points")
        else:
            print(f"    - 5-day: No data")
        
        time.sleep(RATE_LIMIT_DELAY)
        
        # Fetch daily data
        daily_data = fetch_daily_data(symbol, period='2y')
        if daily_data:
            save_cache_file(symbol, 'daily', daily_data)
            results['daily'] = True
            print(f"    ✓ Daily: {len(daily_data['data'])} data points")
        else:
            print(f"    - Daily: No data")
        
        # Mark as successful if at least one data type was saved
        results['success'] = any([results['intraday'], results['5day'], results['daily']])
        
    except Exception as e:
        print(f"    ✗ Error processing {symbol}: {e}")
        
        # Retry logic
        if retry_count < MAX_RETRIES:
            print(f"    ↻ Retrying {symbol} (attempt {retry_count + 1}/{MAX_RETRIES})...")
            time.sleep(2 * RATE_LIMIT_DELAY)  # Wait longer before retry
            return process_stock(symbol, retry_count + 1)
    
    return results


print("✓ Stock processing function defined")

## 8. Batch Process All Stocks

**⚠️ Important Notes:**

1. **Start Small**: The cell below processes only the first 10 stocks by default. Uncomment the full list once you've verified it works.
2. **Time Required**: Processing all ~1860 stocks will take several hours due to API rate limiting.
3. **Progress Saved**: Each stock is cached individually, so you can stop and resume anytime.
4. **API Limits**: Yahoo Finance may rate limit if you go too fast. Adjust `RATE_LIMIT_DELAY` if needed.

In [ ]:
# Test with a small subset first (uncomment to process all stocks)
test_symbols = all_symbols[:10]  # Start with 10 stocks
# test_symbols = all_symbols  # Uncomment this to process ALL stocks

print(f"Starting batch processing for {len(test_symbols)} stocks...")
print(f"Estimated time: ~{len(test_symbols) * 3 * RATE_LIMIT_DELAY / 60:.1f} minutes\n")

# Track results
all_results = []
success_count = 0
failed_symbols = []

# Process each stock with progress bar
for i, symbol in enumerate(tqdm(test_symbols, desc="Processing stocks")):
    print(f"\n[{i+1}/{len(test_symbols)}]", end=" ")
    
    result = process_stock(symbol)
    all_results.append(result)
    
    if result['success']:
        success_count += 1
    else:
        failed_symbols.append(symbol)
    
    # Add a small delay between stocks
    time.sleep(RATE_LIMIT_DELAY)

# Print summary
print(f"\n{'='*60}")
print(f"✓ Batch processing complete!")
print(f"  Success: {success_count}/{len(test_symbols)}")
print(f"  Failed: {len(failed_symbols)}")
if failed_symbols:
    print(f"\n  Failed symbols: {', '.join(failed_symbols[:20])}")
    if len(failed_symbols) > 20:
        print(f"  ... and {len(failed_symbols) - 20} more")
print(f"{'='*60}")

## 9. Verify Generated Cache Files

In [ ]:
# Count cache files
cache_files = list(CACHE_DIR.glob('*.json'))
print(f"✓ Total cache files generated: {len(cache_files)}")

# Group by type
intraday_files = [f for f in cache_files if '_intraday.json' in f.name]
day5_files = [f for f in cache_files if '_5day.json' in f.name]
daily_files = [f for f in cache_files if '_daily.json' in f.name]

print(f"\nBreakdown by type:")
print(f"  Intraday: {len(intraday_files)} files")
print(f"  5-day:    {len(day5_files)} files")
print(f"  Daily:    {len(daily_files)} files")

# Sample a file to verify format
if cache_files:
    sample_file = cache_files[0]
    with open(sample_file, 'r') as f:
        sample_data = json.load(f)
    
    print(f"\n✓ Sample cache file: {sample_file.name}")
    print(f"  Headers: {sample_data['headers']}")
    print(f"  Data points: {len(sample_data['data'])}")
    print(f"  First data point: {sample_data['data'][0]}")
    print(f"  Last data point: {sample_data['data'][-1]}")

## 10. Update Server Code to Use Cache

To use the cached time series data, you need to update the `lib/yahoo_data_source.js` file to return cached data when in CANNED_DATA mode.

Add this code to the `getTimeSeries` function:

```javascript
this.getTimeSeries = function(params, callback) {
  if(/true/i.test(process.env.CANNED_DATA)) {
    var cacheFile = __dirname + '/../public/data/' + process.env.DATA_DOMAIN + 
                    '/cache/' + params.id + '_' + params.type + '.json';
    
    if(fs.existsSync(cacheFile)) {
      fs.readFile(cacheFile, 'utf8', function(err, data) {
        if(err) {
          console.error('Error reading cache file:', cacheFile, err);
          callback(null);
        } else {
          callback(JSON.parse(data));
        }
      });
      return;
    } else {
      console.log('Cache file not found:', cacheFile);
      callback(null);
      return;
    }
  }
  
  // Original code for live data...
  switch(params.type) {
    case 'intraday':
    case '5day': return getIntraday(params, callback);
    case 'daily': return getDaily(params, callback);
    default: callback(null);
  }
};
```

## 11. Next Steps

### To generate cache for ALL stocks:

1. In section 8 above, change:
   ```python
   test_symbols = all_symbols[:10]  # Start with 10 stocks
   ```
   to:
   ```python
   test_symbols = all_symbols  # Process ALL stocks
   ```

2. Run the cell - this will take 2-3 hours for all ~1860 stocks

3. The notebook saves progress as it goes, so you can stop and resume anytime

### Tips for large batches:

- Run overnight or during off-hours
- Monitor for API rate limit errors
- If you get rate limited, increase `RATE_LIMIT_DELAY` to 1.0 or higher
- You can process in chunks by slicing the symbol list:
  ```python
  # Process stocks 100-200
  test_symbols = all_symbols[100:200]
  ```

### Once complete:

1. Update `lib/yahoo_data_source.js` with the code from section 10
2. Restart the server
3. The app will now use your cached time series data! 🎉